In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../datasets")

## Load data

In [2]:
# read the merged dataset, which includes all reviews from all datasets
df = pd.read_csv(DATA_DIR / "merged_dataset.csv", low_memory=False)

In [3]:
# basic stats
print(f"Shape: {df.shape}")
print(f"\nDataset counts:")
print(df["dataset"].value_counts())

Shape: (33079, 30)

Dataset counts:
dataset
pitchfork          22853
critique_brainz    10226
Name: count, dtype: int64


In [4]:
df.head(3)

,dataset,review_id,entity_id,text,rating,album,artist,reviewer_name,reviewer_id,reviewer_type,...,review_url,is_standard_review,pub_date,body,title,score,artist_count,genres,author,cleaned_body
0,critique_brainz,60d4e2a1-53be-4ca9-9683-31db6b7cbe46,08bb6ce3-bec6-4b5e-8c6f-722e45e78fb2,I had the opportunity to attend the digital pr...,NaN,La Pluma o La Espada,Astrid Hadad,jacostamolina,70a41446-ffcf-4ebd-8892-4c3ed06f0774,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,critique_brainz,6424629d-e6b9-410a-8f7e-aa0a6f57951a,642b183a-3e59-3cb9-9187-aeeefa0d8818,Appetite for Destruction is still the better a...,4.0,Use Your Illusion I,Guns N’ Roses,smcamp1234,02fb8586-29e9-4023-9f9a-e659d222210f,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,critique_brainz,abc11944-1e57-4541-beef-23c5b3cf97b9,88727823-999e-45cb-8e43-69b26e9a670e,This is the kind of heartfelt 2010s-style tran...,3.0,AD:TRANCE,Diverse System,yorelkein,5ebf0cdb-c2a8-4170-9e78-a77b7bbd0a93,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Deduplicate albums

In [5]:
# function to normalize album and artist names

import re
import unicodedata


def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):  # curly -> straight apostrophe
        s = s.replace(ch, "'")
    s = s.replace("\u2026", "...").replace("\u00b7", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s)
    return s

In [6]:
pairs = df[["album", "artist"]].dropna()
pairs["artist_norm"] = pairs["artist"].map(normalize)
pairs["album_norm"] = pairs["album"].map(normalize)
pairs.loc[pairs["artist_norm"].isin(["various", "va"]), "artist_norm"] = "various artists"

print(f"Raw unique pairs: {pairs.drop_duplicates(['artist', 'album']).shape[0]:,}")

unique_albums = (
    pairs.groupby(["artist_norm", "album_norm"], as_index=False)
    .agg(
        artist=("artist", "first"),
        album=("album", "first"),
        review_count=("album", "size"),
        n_spellings=("album", "nunique"),
    )
    .sort_values(["artist", "album"])
)

print(f"After dedupe: {len(unique_albums):,}")

Raw unique pairs: 30,735
After dedupe: 30,516


In [7]:
# Checking Beatles to make sure it's correct
beatles = unique_albums.loc[unique_albums["artist_norm"] == "the beatles"]
beatles

,artist_norm,album_norm,artist,album,review_count,n_spellings
24233,the beatles,a hard day's night,The Beatles,A Hard Day’s Night,2,2
24234,the beatles,abbey road,The Beatles,Abbey Road,2,1
24235,the beatles,beatles for sale,The Beatles,Beatles for Sale,3,2
24236,the beatles,help!,The Beatles,Help!,2,1
24237,the beatles,let it be,The Beatles,Let It Be,1,1
24238,the beatles,let it be (super deluxe),The Beatles,Let It Be (Super Deluxe),1,1
24239,the beatles,let it be... naked,The Beatles,Let It Be… Naked,2,2
24240,the beatles,love,The Beatles,Love,2,1
24241,the beatles,magical mystery tour,The Beatles,Magical Mystery Tour,1,1
24242,the beatles,on air - live at the bbc vol. 2,The Beatles,On Air - Live at the BBC Vol. 2,1,1


In [8]:
# cleaning up df
unique_albums = unique_albums[['artist', 'album', 'review_count']]

# using index as album_id
unique_albums["album_id"] = unique_albums.index

# reordering columns
cols = ["album_id"] + [col for col in unique_albums.columns if col != "album_id"]
unique_albums = unique_albums[cols]

# Replace curly apostrophes in 'album' and 'artist' fields with straight ones
unique_albums['album'] = unique_albums['album'].str.replace("’", "'", regex=False)
unique_albums['artist'] = unique_albums['artist'].str.replace("’", "'", regex=False)

In [9]:
unique_albums.head()

,album_id,artist,album,review_count
0,0,!!!,As If,1
1,1,!!!,"Jamie, My Intentions Are Bass EP",1
2,2,!!!,Louden Up Now,1
3,3,!!!,Myth Takes,2
4,4,!!!,Shake the Shudder,1


## Enhanced metadata

Look up each album on MusicBrainz (via Troi's HTTP client) and append release-group metadata to `unique_albums`.

**Run cell 1 (setup), then cell 2 (batch).** Re-run batch to resume — rows with `mb_status` already set are skipped. Progress is checkpointed to `unique_albums_mb.csv`.


In [ ]:
import re
import time

import troi.http_request
from tqdm.auto import tqdm

MB_CACHE_PATH = DATA_DIR / "unique_albums_mb.csv"
MB_COLS = [
    "mb_status",
    "mb_error",
    "mb_release_group_mbid",
    "mb_artist_mbid",
    "release_year",
    "release_type",
    "mb_tags",
    "mb_genre",
]
REQUEST_DELAY_SEC = 1.5
SAVE_EVERY = 20
MAX_ALBUMS = None  # set None to run all pending

In [14]:
def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())


def lookup_mb(artist, album):
    query = f'releasegroup:"{album}" AND artist:"{artist}" AND primarytype:album'
    r = troi.http_request.http_get(
        "https://musicbrainz.org/ws/2/release-group",
        params={"query": query, "fmt": "json", "limit": 5},
    )
    time.sleep(REQUEST_DELAY_SEC)
    r.raise_for_status()
    matches = r.json().get("release-groups", [])
    if not matches:
        return {"mb_status": "not_found", "mb_error": pd.NA}

    target = _norm(album)
    best = min(
        matches,
        key=lambda rg: (0 if _norm(rg.get("title", "")) == target else 1, rg.get("score", 0)),
    )
    mbid = best["id"]

    r = troi.http_request.http_get(
        f"https://musicbrainz.org/ws/2/release-group/{mbid}",
        params={"inc": "tags+genres+artist-credits", "fmt": "json"},
    )
    time.sleep(REQUEST_DELAY_SEC)
    r.raise_for_status()
    detail = r.json()

    year = pd.NA
    if detail.get("first-release-date"):
        year = int(str(detail["first-release-date"])[:4])

    tags = [t["name"] for t in detail.get("tags", [])]
    genres = [g["name"] for g in detail.get("genres", [])]
    artist_mbid = detail["artist-credit"][0]["artist"]["id"]

    return {
        "mb_status": "ok",
        "mb_error": pd.NA,
        "mb_release_group_mbid": mbid,
        "mb_artist_mbid": artist_mbid,
        "release_year": year,
        "release_type": detail.get("primary-type"),
        "mb_tags": "|".join(tags[:20]) if tags else pd.NA,
        "mb_genre": genres[0] if genres else pd.NA,
    }


def mb_pending(status):
    return pd.isna(status) or str(status).strip() == ""

In [15]:

if MB_CACHE_PATH.exists():
    cached = pd.read_csv(MB_CACHE_PATH, low_memory=False)
    unique_albums = unique_albums.drop(
        columns=[c for c in MB_COLS if c in unique_albums.columns],
        errors="ignore",
    )
    unique_albums = unique_albums.merge(
        cached[["album_id", *MB_COLS]],
        on="album_id",
        how="left",
    )
else:
    for col in MB_COLS:
        unique_albums[col] = pd.NA

# clear bogus errors from an earlier missing-import run
bad = unique_albums["mb_error"].astype("string").str.contains("name 'time' is not defined", na=False)
if bad.any():
    unique_albums.loc[bad, MB_COLS] = pd.NA
    print(f"Cleared {bad.sum():,} stale error rows from checkpoint")

unique_albums["mb_status"] = unique_albums["mb_status"].astype("string")
unique_albums["mb_error"] = unique_albums["mb_error"].astype("string")
for col in ["mb_release_group_mbid", "mb_artist_mbid", "release_type", "mb_tags", "mb_genre"]:
    unique_albums[col] = unique_albums[col].astype("string")
unique_albums["release_year"] = pd.to_numeric(unique_albums["release_year"], errors="coerce")

pending = unique_albums.index[unique_albums["mb_status"].map(mb_pending)]
if MAX_ALBUMS is not None:
    pending = pending[:MAX_ALBUMS]

print(f"Pending MB lookups: {len(pending):,} / {len(unique_albums):,}")

Pending MB lookups: 30,466 / 30,516


In [16]:
since_save = 0
for idx in tqdm(pending, desc="MusicBrainz"):
    row = unique_albums.loc[idx]
    try:
        result = lookup_mb(row["artist"], row["album"])
    except Exception as exc:
        result = {"mb_status": "error", "mb_error": str(exc)[:500]}

    for col, value in result.items():
        unique_albums.loc[idx, col] = value

    since_save += 1
    if since_save >= SAVE_EVERY:
        unique_albums.to_csv(MB_CACHE_PATH, index=False)
        since_save = 0

unique_albums.to_csv(MB_CACHE_PATH, index=False)
print("Saved checkpoint.")

MusicBrainz:   0%|          | 0/30466 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
print(unique_albums["mb_status"].value_counts(dropna=False))
unique_albums.loc[unique_albums["mb_status"] == "ok"].head()


mb_status
<NA>         30466
ok              41
not_found        9
Name: count, dtype: int64[pyarrow]


,album_id,artist,album,review_count,mb_status,mb_error,mb_release_group_mbid,mb_artist_mbid,release_year,release_type,mb_tags,mb_genre
0,0,!!!,As If,1,ok,<NA>,85901c1f-b782-44f2-8c76-0016525e2d09,f26c72d3-e52c-467b-b651-679c73d8e1a7,2015.0,Album,dance-punk|electronic|indie rock|rock|tech house,dance-punk
2,2,!!!,Louden Up Now,1,ok,<NA>,79e3ac21-8359-3761-ba35-251a1bd04d68,f26c72d3-e52c-467b-b651-679c73d8e1a7,2004.0,Album,alternative dance|alternative rock|dance-punk|...,alternative dance
3,3,!!!,Myth Takes,2,ok,<NA>,8c1769fe-78f8-3389-9779-8486e446db35,f26c72d3-e52c-467b-b651-679c73d8e1a7,2007.0,Album,acoustic|alternative rock|dance-punk|disco|ele...,alternative rock
4,4,!!!,Shake the Shudder,1,ok,<NA>,0181ab9f-5d34-45c0-bc12-166a39c30f2c,f26c72d3-e52c-467b-b651-679c73d8e1a7,2017.0,Album,rock,rock
5,5,!!!,"Strange Weather, Isn't It?",2,ok,<NA>,4aa3a408-c13e-4e01-a79a-dd1dfecc0d8f,f26c72d3-e52c-467b-b651-679c73d8e1a7,2010.0,Album,electronic|indie|indie rock|rock,electronic


## Subset

For benchmarking, I'm using a small sample of albums that have > 2 reviews.

In [ ]:
unique_albums.loc[unique_albums["review_count"] > 2].to_csv(DATA_DIR / 'albums_3plus.csv', index=False)

In [68]:
unique_albums.loc[unique_albums["review_count"] > 1].to_csv(DATA_DIR / 'albums_2plus.csv', index=False)

In [54]:
# unique_albums.to_csv(DATA_DIR / 'albums.csv', index=False)